# A05 · Deep Networks: Init, Normalization & Residuals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a05_deep_nets.ipynb)

Stacking layers multiplies signals and gradients, so they explode or vanish unless you control their variance. Three tricks make deep nets trainable: variance-preserving initialisation, normalisation and residual connections. Each has a short derivation.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a05). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Derive He initialisation $\mathrm{Var}(w) = 2/n_{in}$ for ReLU networks.
- Compute LayerNorm and RMSNorm by hand and explain why transformers prefer them over BatchNorm.
- Show numerically why residual connections keep gradients alive in deep stacks.
- Use dropout and weight decay correctly (train vs eval mode).

## The math

### He initialisation

$$\mathrm{Var}(z^{(l)}) = n_{in}\,\mathrm{Var}(w)\,\mathbb{E}\!\left[(x^{(l)})^2\right] = n_{in}\,\mathrm{Var}(w)\,\tfrac12\mathrm{Var}(z^{(l-1)})$$

ReLU keeps half the second moment of a symmetric input. Keeping variance constant across layers needs $n_{in}\mathrm{Var}(w)\cdot\tfrac12 = 1$.

**Worked example**

1. $\mathrm{Var}(w) = 2/n_{in}$ (He (Kaiming) init)
2. $n_{in} = 512 \Rightarrow \mathrm{std}(w) = \sqrt{2/512} = 0.0625$
3. With std = 1 instead: each layer multiplies std by $\sqrt{512/2} = 16$ ($16^{32} > 3.4\times10^{38}$: float32 overflows before layer 32)
4. Xavier on ReLU: factor $1/\sqrt2$ per layer $\Rightarrow (0.707)^{50} \approx 3\times10^{-8}$ (the signal vanishes)

### LayerNorm and RMSNorm

$$\mathrm{LN}(x) = \gamma\odot\frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta, \qquad \mathrm{RMSNorm}(x) = \gamma\odot\frac{x}{\sqrt{\tfrac1d\sum_i x_i^2 + \epsilon}}$$

Statistics are over the feature dimension of each token. RMSNorm (Llama) skips the mean, which is cheaper and works as well in practice.

**Worked example**

1. $x = (2, 4, 6, 8):\ \mu = 5,\ \sigma^2 = \tfrac{9+1+1+9}{4} = 5$ (population variance)
2. $\mathrm{LN}(x) = (x - 5)/2.236 = (-1.342,\ -0.447,\ 0.447,\ 1.342)$ (with $\gamma = 1,\ \beta = 0$)
3. $\mathrm{RMS} = \sqrt{(4+16+36+64)/4} = \sqrt{30} = 5.477 \Rightarrow (0.365,\ 0.730,\ 1.095,\ 1.461)$ (RMSNorm keeps the sign pattern)

### Why residuals keep gradients alive

$$x_{L} = x_0 + \sum_{l=0}^{L-1} F_l(x_l) \;\Rightarrow\; \frac{\partial x_L}{\partial x_0} = I + \frac{\partial}{\partial x_0}\sum_l F_l(x_l)$$

Without residuals the gradient is a product of $L$ Jacobians, $\prod_l J_l$: if their norms are about 0.9, after 50 layers you get $0.9^{50} \approx 0.005$.

**Worked example**

1. $0.9^{50} = 0.0052,\quad 1.1^{50} = 117$ (vanish or explode)
2. with residual: $\|I + J\|$ stays near 1 when $F$ starts small (why GPT-2 scales residual-branch init by 1/√(2L))

## Lab: Make a 50-layer network trainable

Reproduce the exploding and vanishing signals, then fix them three ways.

### Step 1

Push a batch through 50 random layers with different inits and record the activation std.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
def depth_stats(init, act=torch.relu, depth=50, width=512):
    torch.manual_seed(0)
    x = torch.randn(1024, width)
    stds = []
    for _ in range(depth):
        W = torch.empty(width, width)
        if init == 'normal': nn.init.normal_(W, std=1.0)
        if init == 'xavier': nn.init.xavier_normal_(W)
        if init == 'he':     nn.init.kaiming_normal_(W, nonlinearity='relu')
        x = act(x @ W)
        stds.append(x.std().item())
    return stds
for init in ['normal', 'xavier', 'he']:
    s = depth_stats(init)
    print(f"{init:7s} layer 1 std={s[0]:.3g}   layer 50 std={s[-1]:.3g}")
# normal -> inf/nan, xavier -> ~1e-8, he -> stays O(1)

### Step 2

LayerNorm and RMSNorm by hand vs the built-in modules.

In [ ]:
x = torch.tensor([[2., 4., 6., 8.]])
mu = x.mean(-1, keepdim=True)
var = x.var(-1, unbiased=False, keepdim=True)
print((x - mu) / torch.sqrt(var + 1e-5))                    # [-1.3416, -0.4472, 0.4472, 1.3416]
print(nn.LayerNorm(4, elementwise_affine=False)(x))
rms = x.pow(2).mean(-1, keepdim=True).add(1e-6).sqrt()
print(x / rms)                                              # [0.3651, 0.7303, 1.0954, 1.4606]

### Step 3

Gradient reaching the first layer of a 30-layer tanh stack, plain vs residual.

In [ ]:
def first_layer_grad(residual, depth=30, d=64):
    torch.manual_seed(0)
    layers = nn.ModuleList([nn.Linear(d, d) for _ in range(depth)])
    h = torch.randn(32, d)
    for lin in layers:
        h = h + torch.tanh(lin(h)) if residual else torch.tanh(lin(h))
    h.pow(2).mean().backward()
    return layers[0].weight.grad.norm().item()
print("plain   ", first_layer_grad(False))
print("residual", first_layer_grad(True))

### Step 4

A reusable pre-LN residual block, and dropout's train/eval behaviour.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, p=0.1):
        super().__init__()
        self.ln = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d), nn.Dropout(p))
    def forward(self, x):
        return x + self.mlp(self.ln(x))
blk = Block(16)
x = torch.randn(2, 16)
blk.train(); print(torch.equal(blk(x), blk(x)))   # False: dropout is random
blk.eval();  print(torch.equal(blk(x), blk(x)))   # True: dropout off at inference

### Clean up

Nothing to clean up.

## Check yourself

**1. Layer with $n_{in} = 1024$, ReLU activation. He-init standard deviation?**

A. $1/32 \approx 0.031$  
B. $\sqrt{2/1024} \approx 0.044$  
C. $2/1024$  
D. $1$

<details><summary>Answer</summary>

**B.** $\mathrm{std} = \sqrt{2/n_{in}} = \sqrt{2/1024} = 0.0442$.

</details>

**2. Why do transformers use LayerNorm rather than BatchNorm?**

A. It's faster on GPUs  
B. It normalises per token, independent of batch size and sequence padding  
C. It has no parameters  
D. BatchNorm doesn't support backprop

<details><summary>Answer</summary>

**B.** LayerNorm statistics are computed over features within each token, so they're identical in training and inference and unaffected by batch composition.

</details>

**3. In $y = x + F(x)$, what is $\partial y/\partial x$?**

A. $\partial F/\partial x$  
B. $I + \partial F/\partial x$  
C. $I$  
D. $x\,\partial F/\partial x$

<details><summary>Answer</summary>

**B.** The identity term guarantees a direct gradient path to earlier layers.

</details>